***Dependencies***

In [ ]:
# import sys
# !{sys.executable} -m pip install ucimlrepo
# !{sys.executable} -m pip install openpyxl

In [101]:
import pandas as pd
import numpy as np
from pathlib import Path
from config import get_rng, count_missing, DISEASES, CATEGORICAL_FEATURES
from ucimlrepo import fetch_ucirepo 
import openpyxl
from sklearn.preprocessing import LabelEncoder

rng = get_rng()

ORIGINAL_DATA_DIR = Path("original_datasets")
ORIGINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_DIR = Path("processed_datasets")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

***Saving original data***

In [90]:
#fetchable datasets

breast_cancer_dataset = fetch_ucirepo(id=15) 
obesity_dataset = fetch_ucirepo(id=544)
hepatitis_dataset = fetch_ucirepo(id=503)
eeg_eye_dataset = fetch_ucirepo(id=264) 
diabetes_dataset = fetch_ucirepo(id=891) 
  
breast_cancer_df = breast_cancer_dataset.data.original
breast_cancer_path = ORIGINAL_DATA_DIR / f"breast_cancer.csv"
obesity_df = obesity_dataset.data.original
obesity_path = ORIGINAL_DATA_DIR / f"obesity.csv"
hepatitis_df = hepatitis_dataset.data.original
hepatitis_path = ORIGINAL_DATA_DIR / f"hepatitis.csv"
eeg_eye_df = eeg_eye_dataset.data.original
eeg_eye_path = ORIGINAL_DATA_DIR / f"eeg_eye.csv"
diabetes_df = diabetes_dataset.data.original
diabetes_path = ORIGINAL_DATA_DIR / f"diabetes.csv"


breast_cancer_df.to_csv(breast_cancer_path, index=False)
print(f"Saved: {breast_cancer_path}")
obesity_df.to_csv(obesity_path, index=False)
print(f"Saved: {obesity_path}")
hepatitis_df.to_csv(hepatitis_path, index=False)
print(f"Saved: {hepatitis_path}")
eeg_eye_df.to_csv(eeg_eye_path, index=False)
print(f"Saved: {eeg_eye_path}")
diabetes_df.to_csv(diabetes_path, index=False)
print(f"Saved: {diabetes_path}")

Saved: original_datasets\breast_cancer.csv
Saved: original_datasets\obesity.csv
Saved: original_datasets\hepatitis.csv
Saved: original_datasets\eeg_eye.csv
Saved: original_datasets\diabetes.csv


In [ ]:
#non-fetchable dataset

## read csv
# NF1_df = pd.read_csv('original_datasets/NF1.csv')

## read xlsx (original format)
wb = openpyxl.load_workbook('original_datasets/NF1.xlsx')
ws = wb['Dataset']
NF1_df = pd.DataFrame(ws.values)
nf1_path = ORIGINAL_DATA_DIR / f"NF1.csv"
NF1_df.to_csv(nf1_path, index=False)
print(f"Saved: {nf1_path}")

***Deleteing non usable rows and columns + setting target as most right column***

In [92]:
# breast cancer - deleting first column (ID)
processed_breast_cancer_df = breast_cancer_df.iloc[:, 1:]

In [93]:
# diabetes - deleting first column (ID), moving target to last column and picking random sample of 30000 rows
processed_diabetes_df = diabetes_df.iloc[:, 1:]

processed_diabetes_df = processed_diabetes_df[[col for col in processed_diabetes_df.columns if col != 'Diabetes_binary'] + ['Diabetes_binary']]

random_state = rng.integers(low=0, high=10000)
processed_diabetes_df = processed_diabetes_df.sample(n=30000, random_state=random_state)

In [94]:
# NF1 - deleting first row and column because of bad formatting of original file, moving target to last column

processed_NF1_df = NF1_df.copy()
processed_NF1_df.columns = processed_NF1_df.iloc[0]
processed_NF1_df = processed_NF1_df.iloc[1:, 1:]
processed_NF1_df = processed_NF1_df.reset_index(drop=True)

processed_NF1_df = processed_NF1_df[[col for col in processed_NF1_df.columns if col != "Case Type"] + ["Case Type"]]

In [95]:
# obesity

processed_obesity_df = obesity_df.copy()
for col in processed_obesity_df.columns:
    if processed_obesity_df[col].dtype == 'float64' and col in CATEGORICAL_FEATURES["obesity"]:
        min = processed_obesity_df[col].min()
        max = processed_obesity_df[col].max()
        processed_obesity_df[col] = processed_obesity_df[col].clip(min, max).astype(int)


***Ensuring uniform baseline for datasets with syteteic missingness***

In [103]:
miss_data = []
dataframes = [processed_breast_cancer_df, processed_diabetes_df, eeg_eye_df, hepatitis_df, processed_NF1_df, processed_obesity_df]
for df, name in zip(dataframes, DISEASES):
    total, missing_count, missing_percentage = count_missing(df)
    total_cols = df.shape[1]
    miss_cols = df.isna().any(axis=0).sum()
    total_rows = df.shape[0]
    miss_rows = df.isna().any(axis=1).sum()
    miss_data.append({
        "file": name,
        "total_data_points": total,
        "missing_count": missing_count,
        "missing_percentage": missing_percentage,
        "total_cols": total_cols,
        "miss_cols": miss_cols,
        "total_rows": total_rows,
        "miss_rows": miss_rows
    })
    
data_df = pd.DataFrame(miss_data)
data_df

,file,total_data_points,missing_count,missing_percentage,total_cols,miss_cols,total_rows,miss_rows
0,breast_cancer,6291,16,0.254332,10,1,699,16
1,diabetes,630000,0,0.000000,22,0,30000,0
2,eeg_eye,209720,0,0.000000,15,0,14980,0
3,hepatitis,38780,0,0.000000,29,0,1385,0
4,NF1,5328,73,1.370120,19,3,296,38
5,obesity,33776,0,0.000000,17,0,2111,0


In [104]:
dataframes_names = dict(zip(DISEASES, dataframes))
clean_df = dataframes_names["breast_cancer"].copy()
clean_df = clean_df.dropna(how='any')
dataframes_names["breast_cancer"] = clean_df

clean_df = dataframes_names["NF1"].copy()
clean_df = clean_df.dropna(how='any')
dataframes_names["NF1"] = clean_df

In [105]:
miss_data = []
for name, df in dataframes_names.items():
    total, missing_count, missing_percentage = count_missing(df)
    total_cols = df.shape[1]
    miss_cols = df.isna().any(axis=0).sum()
    total_rows = df.shape[0]
    miss_rows = df.isna().any(axis=1).sum()
    miss_data.append({
        "file": name,
        "total_data_points": total,
        "missing_count": missing_count,
        "missing_percentage": missing_percentage,
        "total_cols": total_cols,
        "miss_cols": miss_cols,
        "total_rows": total_rows,
        "miss_rows": miss_rows
    })
    
data_df = pd.DataFrame(miss_data)
data_df

,file,total_data_points,missing_count,missing_percentage,total_cols,miss_cols,total_rows,miss_rows
0,breast_cancer,6147,0,0.0,10,0,683,0
1,diabetes,630000,0,0.0,22,0,30000,0
2,eeg_eye,209720,0,0.0,15,0,14980,0
3,hepatitis,38780,0,0.0,29,0,1385,0
4,NF1,4644,0,0.0,19,0,258,0
5,obesity,33776,0,0.0,17,0,2111,0


***Encoding categorical data and normalising names of variables***


In [106]:
encoded_datasets = []

for name, df in dataframes_names.items():
    le = LabelEncoder()
    df_encoded = df.copy()
    df_encoded.columns = df_encoded.columns.str.strip().str.replace(' ', '_')
    for col in CATEGORICAL_FEATURES[name]:
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    df_encoded.iloc[:, -1] = le.fit_transform(df_encoded.iloc[:, -1].astype(str))
    encoded_datasets.append((name, df_encoded))

***Saving preprocessed data***

In [107]:
for name, df_encoded in encoded_datasets:
    output_path = PROCESSED_DATA_DIR / f"{name}.csv"
    df_encoded.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

Saved: processed_datasets\breast_cancer.csv
Saved: processed_datasets\diabetes.csv
Saved: processed_datasets\eeg_eye.csv
Saved: processed_datasets\hepatitis.csv
Saved: processed_datasets\NF1.csv
Saved: processed_datasets\obesity.csv


***Target class balance***

In [108]:
target_count_data = []
for name, df_encoded in encoded_datasets:
    target_count = df_encoded.iloc[:, -1].value_counts()
    tmp_data = { "file" : name }
    for value in sorted(target_count.index):
        tmp_data[value] = target_count[value]
    target_count_data.append(tmp_data)

target_count_data_df = pd.DataFrame(target_count_data)
target_count_data_df

,file,0,1,2,3,4,5,6
0,breast_cancer,444,239,NaN,NaN,NaN,NaN,NaN
1,diabetes,25829,4171,NaN,NaN,NaN,NaN,NaN
2,eeg_eye,8257,6723,NaN,NaN,NaN,NaN,NaN
3,hepatitis,336,332,355.0,362.0,NaN,NaN,NaN
4,NF1,139,119,NaN,NaN,NaN,NaN,NaN
5,obesity,272,287,351.0,297.0,324.0,290.0,290.0
